# Lex-Indic — QLoRA fine-tune on Kaggle (free GPU)

Trains a small base model on the Lex-Indic BNS/BNSS/BSA instruction data and
exports a **GGUF** that drops into the app's Ollama path with zero code change
(`LLM_PROVIDER=ollama OLLAMA_MODEL=lexindic-qwen`).

**Before you run:**
1. Kaggle → Settings → **phone-verify** your account (needed for GPU + Internet).
2. This notebook: right sidebar → **Accelerator = GPU T4 x2**, **Internet = On**.
3. Upload `data/finetune/train.jsonl` as a Kaggle **Dataset** (e.g. name it
   `lexindic-train`) and **Add** it to this notebook. It will appear at
   `/kaggle/input/lexindic-train/train.jsonl`.
4. Run all cells. Per the sprint plan, run the notebook **twice** — once with
   `BASE = "qwen"`, once with `BASE = "llama"` — and let `tools/eval_model.py`
   pick the winner.

Cost: **0.** One run is ~1-3 h on a single T4; free quota is 30 h/week.

## 1. Config — pick the base model

In [ ]:
# Run once as 'qwen', once as 'llama'.
BASE = "qwen"   # or "llama"

BASE_MODELS = {
    "qwen":  "unsloth/Qwen2.5-7B-Instruct",       # better Devanagari/Hindi
    "llama": "unsloth/Meta-Llama-3.1-8B-Instruct", # widest tooling
}
CHAT_TEMPLATE = {"qwen": "qwen-2.5", "llama": "llama-3.1"}[BASE]
OUT_NAME  = f"lexindic-{BASE}"
TRAIN_JSONL = None  # auto-discovered in the sanity-check cell below

EPOCHS = 3.0
MAX_SEQ_LEN = 2048
LR = 2e-4
QUANT = "q4_k_m"   # GGUF quant Ollama serves
print(f"Base: {BASE_MODELS[BASE]}  ->  {OUT_NAME}.{QUANT}.gguf")

## 2. Install Unsloth (2x faster QLoRA + one-call GGUF export)

Kaggle's image already ships a working torch + CUDA, so we install Unsloth
*on top* rather than letting it re-pin torch (that's what breaks Kaggle runs).
**No `%%capture`** — if an install fails we need to see it immediately, not
discover it three cells later as a confusing ImportError.

In [ ]:
!pip install -q --upgrade unsloth unsloth_zoo
!pip install -q --no-deps trl peft accelerate bitsandbytes

### 2b. Verify the environment before spending GPU time

If anything is wrong, it fails **here** in ~1 minute instead of after a long run.

In [ ]:
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "No GPU. Right sidebar -> Accelerator -> 'GPU T4 x2', then re-run.")
print("GPU:", torch.cuda.get_device_name(0))

from unsloth import FastLanguageModel
print("unsloth import: OK")

## 3. Sanity-check the training data

In [ ]:
import json, os, glob

# Auto-discover the uploaded .jsonl anywhere under /kaggle/input, so the exact
# dataset folder / file name need not match — it just has to be attached.
found = glob.glob('/kaggle/input/**/*.jsonl', recursive=True)
print('under /kaggle/input:', os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'NONE ATTACHED')
print('.jsonl found:', found)
assert found, ('No .jsonl under /kaggle/input. Right panel -> Add Input -> '
               'attach lexindic-train, then re-run.')
train = [f for f in found if os.path.basename(f) == 'train.jsonl'] \
        or sorted(found, key=os.path.getsize, reverse=True)
TRAIN_JSONL = train[0]
print('Using:', TRAIN_JSONL)

rows = [json.loads(l) for l in open(TRAIN_JSONL, encoding='utf-8')]
assert rows and 'messages' in rows[0], "Expected chat-format {'messages':[...]} rows."
hindi = sum(1 for r in rows if any(ord(c) > 0x900 and ord(c) < 0x980
            for m in r['messages'] for c in m['content']))
print(f"{len(rows)} training pairs  ({hindi} contain Devanagari)")
print('sample Q:', rows[0]['messages'][-2]['content'][:120])
print('sample A:', rows[0]['messages'][-1]['content'][:120])

## 4. Load base model in 4-bit (QLoRA)

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODELS[BASE],
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,     # 4-bit base + trainable LoRA adapters
    dtype=None,
)
tokenizer = get_chat_template(tokenizer, chat_template=CHAT_TEMPLATE)

model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=7,
)

## 5. Train

Same hyperparameters as `training/train_qlora.py` so the Kaggle run and the
documented RunPod run are identical. ~1-3 h on one T4.

In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

def fmt(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False,
                                                   add_generation_prompt=False)
                     for m in batch["messages"]]}

ds = load_dataset("json", data_files=TRAIN_JSONL, split="train").map(fmt, batched=True)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        warmup_ratio=0.05,
        logging_steps=10,
        optim="adamw_8bit",
        seed=7,
        output_dir=f"{OUT_NAME}-ckpt",
    ),
)
trainer.train()

## 6. Smoke-test the fine-tune (held-out-style question)

Ask it a mapping it should now know cold. Watch for the **right** BNS number
and no invented section.

In [ ]:
FastLanguageModel.for_inference(model)
msgs = [{"role": "user",
         "content": "Which BNS section replaces IPC 304B (dowry death), and what is the punishment?"}]
inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                       return_tensors="pt").to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.2)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))
# Expected: BNS Section 80, minimum 7 years to life. (NOT 84.)

## 7. Export GGUF for Ollama

One call: merges LoRA into the base and writes a `q4_k_m` GGUF (~4-5 GB).
Unsloth builds llama.cpp under the hood the first time (a few minutes).

In [ ]:
import os, glob, shutil, gc, torch

# Insurance FIRST: save the small LoRA adapter so a GGUF/disk failure never
# wastes the ~1h training run.
model.save_pretrained('/kaggle/working/lexindic-lora')
tokenizer.save_pretrained('/kaggle/working/lexindic-lora')
print('LoRA adapter saved to /kaggle/working/lexindic-lora')

# Free disk before the heavy 16-bit merge + GGUF build.
shutil.rmtree(f'{OUT_NAME}-ckpt', ignore_errors=True)
gc.collect(); torch.cuda.empty_cache()

# Build the GGUF in /tmp (NOT capped by the ~20GB /kaggle/working quota),
# then copy only the final ~5GB file back for download.
try:
    BUILD = f'/tmp/{OUT_NAME}_gguf'
    model.save_pretrained_gguf(BUILD, tokenizer, quantization_method=QUANT)
    g = sorted(glob.glob(f'{BUILD}/*.gguf') + glob.glob(f'{BUILD}*.gguf'))
    assert g, 'no gguf produced'
    dst = f'/kaggle/working/{OUT_NAME}.{QUANT}.gguf'
    shutil.copy(g[0], dst)
    print('GGUF ready:', dst, round(os.path.getsize(dst) / 1e9, 1), 'GB')
except Exception as e:
    print('GGUF export failed:', repr(e))
    print('Adapter saved in /kaggle/working/lexindic-lora - convert to GGUF locally.')

## 8. Download + serve locally

1. Right pane → **Output** → download `lexindic-<base>.q4_k_m.gguf`.
2. On your Mac, put it next to `training/Modelfile.<base>` (edit the `FROM`
   line to the exact filename), then:
   ```bash
   cd training
   ollama create lexindic-qwen -f Modelfile.qwen   # or Modelfile.llama
   ```
3. Score it against the Groq baseline on the held-out gold set:
   ```bash
   python3 tools/eval_model.py --models groq,ollama:lexindic-qwen,ollama:lexindic-llama
   ```
4. Serve the winner air-gapped — zero app-code change:
   ```bash
   LLM_PROVIDER=ollama OLLAMA_MODEL=lexindic-qwen python3 app.py
   ```

The eval report (`data/finetune/eval_report.md`) is the sales artifact: the
generic 70B's weak baseline vs the fine-tune's ~95% on held-out BNS mappings.